<a href="https://colab.research.google.com/github/RudraPramanik/ev-battery/blob/supply-chain-model/supply_chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Supply Chain Management with Large Language Models

# ============================================
# CELL 1: Installation and Setup
# ============================================
!pip install -q transformers torch pandas numpy scikit-learn
!pip install -q openai langchain chromadb sentence-transformers
!pip install -q plotly dash prophet xgboost
!pip install -q kaggle opendatasets

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
import requests
from typing import Dict, List, Tuple, Optional
import os

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Setup complete!")

# ============================================
# CELL 2: Download Open-Source Datasets
# ============================================
# DataCo Supply Chain Dataset
!wget -q https://raw.githubusercontent.com/supplychainpy/supplychainpy/master/supplychainpy/sample_data/complete_dataset_v2.csv -O supply_chain_data.csv

# Brazilian E-commerce Dataset (Olist)
!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -q brazilian-ecommerce.zip

# Additional synthetic data generation for demonstration
def generate_synthetic_supply_data(n_samples=10000):
    """Generate synthetic supply chain data for demonstration"""
    np.random.seed(42)

    data = {
        'order_id': range(1, n_samples + 1),
        'date': pd.date_range(start='2022-01-01', periods=n_samples, freq='H'),
        'product_id': np.random.choice(['P001', 'P002', 'P003', 'P004', 'P005'], n_samples),
        'quantity': np.random.poisson(20, n_samples),
        'unit_price': np.random.uniform(10, 100, n_samples),
        'supplier_id': np.random.choice(['S001', 'S002', 'S003', 'S004'], n_samples),
        'warehouse_id': np.random.choice(['W001', 'W002', 'W003'], n_samples),
        'shipping_days': np.random.poisson(5, n_samples),
        'customer_segment': np.random.choice(['Retail', 'Wholesale', 'Online'], n_samples),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n_samples)
    }

    df = pd.DataFrame(data)

    # Add seasonal patterns
    df['season_factor'] = np.sin(2 * np.pi * df.index / 365) + 1.5
    df['quantity'] = (df['quantity'] * df['season_factor']).astype(int)

    # Add trend
    df['trend'] = df.index / 1000
    df['quantity'] = (df['quantity'] * (1 + df['trend'] * 0.1)).astype(int)

    # Calculate total value
    df['total_value'] = df['quantity'] * df['unit_price']

    # Add risk factors
    df['supplier_reliability'] = np.random.uniform(0.7, 1.0, n_samples)
    df['delivery_risk'] = np.random.choice(['Low', 'Medium', 'High'], n_samples, p=[0.7, 0.25, 0.05])

    return df

# Generate and save synthetic data
synthetic_df = generate_synthetic_supply_data(10000)
synthetic_df.to_csv('synthetic_supply_chain.csv', index=False)

print("✅ Datasets loaded successfully!")
print(f"Synthetic data shape: {synthetic_df.shape}")
print(synthetic_df.head())

# ============================================
# CELL 3: Data Processing Pipeline
# ============================================
class SupplyChainDataProcessor:
    """Comprehensive data processor for supply chain data"""

    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}

    def preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """Preprocess supply chain data"""
        df = df.copy()

        # Handle datetime
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
            df['year'] = df['date'].dt.year
            df['month'] = df['date'].dt.month
            df['day'] = df['date'].dt.day
            df['weekday'] = df['date'].dt.weekday
            df['quarter'] = df['date'].dt.quarter

        # Handle categorical variables
        categorical_cols = df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if col not in self.label_encoders:
                self.label_encoders[col] = LabelEncoder()
                df[col + '_encoded'] = self.label_encoders[col].fit_transform(df[col])
            else:
                df[col + '_encoded'] = self.label_encoders[col].transform(df[col])

        # Scale numerical features
        numerical_cols = df.select_dtypes(include=[np.number]).columns
        df[numerical_cols] = self.scaler.fit_transform(df[numerical_cols])

        return df

    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create additional features for ML models"""
        df = df.copy()

        # Lag features for time series
        if 'quantity' in df.columns:
            for lag in [1, 7, 14, 30]:
                df[f'quantity_lag_{lag}'] = df['quantity'].shift(lag)

        # Rolling statistics
        if 'total_value' in df.columns:
            for window in [7, 14, 30]:
                df[f'value_rolling_mean_{window}'] = df['total_value'].rolling(window).mean()
                df[f'value_rolling_std_{window}'] = df['total_value'].rolling(window).std()

        # Interaction features
        if 'quantity' in df.columns and 'unit_price' in df.columns:
            df['price_quantity_interaction'] = df['quantity'] * df['unit_price']

        return df.fillna(method='ffill').fillna(0)

# Process the data
processor = SupplyChainDataProcessor()
processed_df = processor.preprocess(synthetic_df)
feature_df = processor.create_features(processed_df)

print("✅ Data processing complete!")
print(f"Processed data shape: {feature_df.shape}")

# ============================================
# CELL 4: LLM Integration for Supply Chain
# ============================================
class SupplyChainLLM:
    """Supply Chain Management with LLM Integration"""

    def __init__(self, model_name: str = "microsoft/phi-2"):
        """Initialize with open-source model"""
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # we'll use a smaller model that works smoothly in Colab free tier
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                low_cpu_mem_usage=True
            )
            self.model.to(self.device)
            print(f"✅ Loaded {model_name}")
        except Exception as e:
            print(f"⚠️ Could not load {model_name}, using mock mode for demonstration")
            self.model = None

    def generate_response(self, prompt: str, max_length: int = 200) -> str:
        """Generate response from LLM"""
        if self.model is None:
            # Mock response for demonstration
            return self._mock_response(prompt)

        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split(prompt)[-1].strip()

    def _mock_response(self, prompt: str) -> str:
        """Mock responses for demonstration when model can't be loaded"""
        if "demand forecast" in prompt.lower():
            return "Based on historical patterns and market conditions, expected demand: 450 units (±50)"
        elif "supplier risk" in prompt.lower():
            return "Risk Level: MEDIUM - Geopolitical factors and recent performance indicate 65% reliability score"
        elif "inventory optimization" in prompt.lower():
            return "Recommended: Increase safety stock by 15%, implement JIT for fast-moving items"
        else:
            return "Analysis complete. Recommendation: Monitor closely and adjust parameters."

    def analyze_demand(self, historical_data: pd.DataFrame, context: str = "") -> Dict:
        """Analyze demand using LLM + traditional methods"""
        # Traditional statistical analysis
        recent_avg = historical_data['quantity'].tail(30).mean()
        trend = np.polyfit(range(len(historical_data)), historical_data['quantity'], 1)[0]

        # LLM analysis
        prompt = f"""
        Analyze supply chain demand:
        Recent average: {recent_avg:.2f} units
        Trend: {'+' if trend > 0 else ''}{trend:.2f} units/period
        Context: {context}

        Provide demand forecast and confidence level:
        """

        llm_response = self.generate_response(prompt)

        return {
            'statistical_forecast': recent_avg * (1 + trend/100),
            'trend': trend,
            'llm_analysis': llm_response,
            'confidence': 0.75  # Would be calculated based on model confidence
        }

    def assess_supplier_risk(self, supplier_data: Dict, market_conditions: str = "") -> Dict:
        """Assess supplier risk using LLM reasoning"""
        prompt = f"""
        Evaluate supplier risk:
        Supplier ID: {supplier_data.get('supplier_id', 'Unknown')}
        Reliability Score: {supplier_data.get('reliability', 0.0):.2%}
        Recent Delays: {supplier_data.get('delays', 0)}
        Market Conditions: {market_conditions}

        Provide risk assessment and mitigation strategies:
        """

        risk_analysis = self.generate_response(prompt)

        # Calculate risk score
        risk_score = 1.0 - supplier_data.get('reliability', 0.5)
        risk_level = 'HIGH' if risk_score > 0.7 else 'MEDIUM' if risk_score > 0.3 else 'LOW'

        return {
            'risk_score': risk_score,
            'risk_level': risk_level,
            'llm_assessment': risk_analysis,
            'mitigation_strategies': self._generate_mitigation_strategies(risk_level)
        }

    def _generate_mitigation_strategies(self, risk_level: str) -> List[str]:
        """Generate risk mitigation strategies"""
        strategies = {
            'HIGH': [
                'Identify alternative suppliers immediately',
                'Increase safety stock by 30%',
                'Implement daily monitoring',
                'Prepare contingency contracts'
            ],
            'MEDIUM': [
                'Maintain backup supplier list',
                'Increase safety stock by 15%',
                'Weekly performance reviews',
                'Strengthen communication channels'
            ],
            'LOW': [
                'Continue regular monitoring',
                'Maintain standard safety stock',
                'Monthly performance reviews',
                'Build long-term partnership'
            ]
        }
        return strategies.get(risk_level, ['Monitor situation'])

    def optimize_inventory(self, current_inventory: Dict, forecast: Dict, constraints: Dict) -> Dict:
        """Optimize inventory levels using LLM + optimization algorithms"""
        prompt = f"""
        Optimize inventory levels:
        Current Stock: {current_inventory}
        Demand Forecast: {forecast}
        Constraints: {constraints}

        Recommend optimal inventory strategy:
        """

        llm_recommendation = self.generate_response(prompt)

        # Simple EOQ calculation for demonstration
        if 'demand' in forecast and 'holding_cost' in constraints:
            eoq = np.sqrt(2 * forecast['demand'] * constraints.get('ordering_cost', 100) / constraints['holding_cost'])
        else:
            eoq = 100  # Default

        return {
            'eoq': eoq,
            'reorder_point': forecast.get('demand', 0) * constraints.get('lead_time', 5) / 30,
            'safety_stock': forecast.get('demand', 0) * 0.2,  # 20% safety stock
            'llm_strategy': llm_recommendation
        }

# Initialize the LLM system
llm_system = SupplyChainLLM()

# ============================================
# CELL 5: Traditional ML Models for Comparison
# ============================================
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from prophet import Prophet
import xgboost as xgb

class TraditionalMLModels:
    """Traditional ML models for supply chain forecasting"""

    def __init__(self):
        self.models = {
            'linear': LinearRegression(),
            'rf': RandomForestRegressor(n_estimators=100, random_state=42),
            'gb': GradientBoostingRegressor(n_estimators=100, random_state=42),
            'xgb': xgb.XGBRegressor(n_estimators=100, random_state=42)
        }
        self.prophet_model = None

    def train_all(self, X_train, y_train):
        """Train all models"""
        trained_models = {}
        for name, model in self.models.items():
            model.fit(X_train, y_train)
            trained_models[name] = model
            print(f"✅ Trained {name}")
        return trained_models

    def train_prophet(self, df):
        """Train Prophet model for time series"""
        prophet_df = df[['date', 'quantity']].copy()
        prophet_df.columns = ['ds', 'y']

        self.prophet_model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False
        )
        self.prophet_model.fit(prophet_df)
        print("✅ Trained Prophet model")

    def predict_prophet(self, periods=30):
        """Make predictions with Prophet"""
        if self.prophet_model is None:
            return None

        future = self.prophet_model.make_future_dataframe(periods=periods)
        forecast = self.prophet_model.predict(future)
        return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

# Prepare data for ML models
feature_cols = [col for col in feature_df.columns if col not in ['date', 'order_id', 'total_value']]
feature_cols = [col for col in feature_cols if not col.startswith('quantity_lag')]  # Remove for initial training

X = feature_df[feature_cols].fillna(0)
y = feature_df['quantity'].fillna(0)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train traditional models
ml_models = TraditionalMLModels()
trained_models = ml_models.train_all(X_train, y_train)

# Train Prophet
ml_models.train_prophet(synthetic_df)

print("✅ All traditional models trained!")

# ============================================
# CELL 6: Ensemble Model (LLM + Traditional ML)
# ============================================
class EnsembleSupplyChainModel:
    """Ensemble model combining LLM and traditional ML"""

    def __init__(self, llm_system, ml_models):
        self.llm = llm_system
        self.ml_models = ml_models
        self.weights = {'llm': 0.3, 'ml': 0.7}  # Can be optimized

    def predict_demand(self, X, historical_data, context=""):
        """Ensemble demand prediction"""
        predictions = {}

        # Traditional ML predictions
        for name, model in self.ml_models.items():
            predictions[name] = model.predict(X)

        # LLM prediction
        llm_result = self.llm.analyze_demand(historical_data, context)
        predictions['llm'] = llm_result['statistical_forecast']

        # Weighted ensemble
        ml_avg = np.mean([predictions[m] for m in self.ml_models.keys()])
        ensemble_pred = (self.weights['ml'] * ml_avg +
                        self.weights['llm'] * predictions['llm'])

        return {
            'ensemble': ensemble_pred,
            'individual': predictions,
            'llm_analysis': llm_result['llm_analysis']
        }

    def evaluate(self, X_test, y_test, historical_data):
        """Evaluate ensemble model"""
        results = {}

        # Get predictions
        ensemble_results = self.predict_demand(X_test, historical_data)

        # Calculate metrics for each model
        for name, pred in ensemble_results['individual'].items():
            if isinstance(pred, (int, float)):
                pred = np.full(len(y_test), pred)

            results[name] = {
                'rmse': np.sqrt(mean_squared_error(y_test, pred[:len(y_test)])),
                'mae': mean_absolute_error(y_test, pred[:len(y_test)])
            }

        # Ensemble metrics
        ensemble_pred = ensemble_results['ensemble']
        if isinstance(ensemble_pred, (int, float)):
            ensemble_pred = np.full(len(y_test), ensemble_pred)

        results['ensemble'] = {
            'rmse': np.sqrt(mean_squared_error(y_test, ensemble_pred[:len(y_test)])),
            'mae': mean_absolute_error(y_test, ensemble_pred[:len(y_test)])
        }

        return results

# Create and evaluate ensemble model
ensemble = EnsembleSupplyChainModel(llm_system, trained_models)
evaluation_results = ensemble.evaluate(X_test, y_test, synthetic_df)

print("📊 Model Performance Comparison:")
for model_name, metrics in evaluation_results.items():
    print(f"{model_name}: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}")

# ============================================
# CELL 7: Visualization Dashboard
# ============================================
def create_supply_chain_dashboard(df, predictions, evaluation_results):
    """Create interactive dashboard for supply chain analytics"""

    # Create subplots
    fig = go.Figure()

    # 1. Demand Over Time
    fig.add_trace(go.Scatter(
        x=df['date'],
        y=df['quantity'],
        mode='lines',
        name='Actual Demand',
        line=dict(color='blue')
    ))

    # 2. Model Performance Comparison
    model_names = list(evaluation_results.keys())
    rmse_values = [evaluation_results[m]['rmse'] for m in model_names]

    fig_perf = go.Figure([go.Bar(
        x=model_names,
        y=rmse_values,
        text=[f'{v:.2f}' for v in rmse_values],
        textposition='auto',
    )])
    fig_perf.update_layout(
        title='Model Performance Comparison (RMSE)',
        xaxis_title='Model',
        yaxis_title='RMSE',
        showlegend=False
    )

    # 3. Supplier Risk Matrix
    suppliers = df['supplier_id'].unique()
    risk_scores = np.random.uniform(0.2, 0.9, len(suppliers))
    performance_scores = np.random.uniform(0.5, 1.0, len(suppliers))

    fig_risk = go.Figure(data=go.Scatter(
        x=performance_scores,
        y=risk_scores,
        mode='markers+text',
        marker=dict(
            size=20,
            color=risk_scores,
            colorscale='RdYlGn_r',
            showscale=True,
            colorbar=dict(title="Risk Level")
        ),
        text=suppliers,
        textposition="top center"
    ))

    fig_risk.update_layout(
        title='Supplier Risk Assessment Matrix',
        xaxis_title='Performance Score',
        yaxis_title='Risk Score',
        showlegend=False
    )

    # Add quadrant lines
    fig_risk.add_hline(y=0.5, line_dash="dash", line_color="gray")
    fig_risk.add_vline(x=0.75, line_dash="dash", line_color="gray")

    # 4. Inventory Optimization
    products = df['product_id'].unique()
    current_stock = np.random.randint(100, 500, len(products))
    optimal_stock = current_stock * np.random.uniform(0.8, 1.2, len(products))

    fig_inventory = go.Figure()
    fig_inventory.add_trace(go.Bar(name='Current Stock', x=products, y=current_stock))
    fig_inventory.add_trace(go.Bar(name='Optimal Stock', x=products, y=optimal_stock))
    fig_inventory.update_layout(
        title='Inventory Optimization Recommendations',
        xaxis_title='Product',
        yaxis_title='Stock Level',
        barmode='group'
    )

    return fig, fig_perf, fig_risk, fig_inventory

# Create visualizations
fig_demand, fig_performance, fig_risk, fig_inventory = create_supply_chain_dashboard(
    synthetic_df, None, evaluation_results
)

# Display plots
fig_demand.show()
fig_performance.show()
fig_risk.show()
fig_inventory.show()

# ============================================
# CELL 8: Real-time Decision Support System
# ============================================
class SupplyChainDecisionSupport:
    """Real-time decision support system"""

    def __init__(self, llm_system, ensemble_model):
        self.llm = llm_system
        self.ensemble = ensemble_model

    def analyze_scenario(self, scenario: Dict) -> Dict:
        """Analyze a supply chain scenario and provide recommendations"""

        recommendations = {
            'timestamp': datetime.now().isoformat(),
            'scenario': scenario,
            'analysis': {},
            'recommendations': []
        }

        # Analyze demand impact
        if 'demand_shock' in scenario:
            demand_analysis = self.llm.analyze_demand(
                pd.DataFrame({'quantity': [100, 120, 110, scenario['demand_shock']]}),
                f"Sudden demand change to {scenario['demand_shock']} units"
            )
            recommendations['analysis']['demand'] = demand_analysis

        # Assess supplier risk
        if 'supplier_issue' in scenario:
            risk_assessment = self.llm.assess_supplier_risk(
                scenario['supplier_issue'],
                scenario.get('market_conditions', 'Normal')
            )
            recommendations['analysis']['supplier_risk'] = risk_assessment

        # Optimize inventory response
        if 'inventory_constraint' in scenario:
            inventory_optimization = self.llm.optimize_inventory(
                scenario.get('current_inventory', {}),
                scenario.get('forecast', {'demand': 100}),
                scenario['inventory_constraint']
            )
            recommendations['analysis']['inventory'] = inventory_optimization

        # Generate action recommendations
        recommendations['recommendations'] = self._generate_recommendations(
            recommendations['analysis']
        )

        return recommendations

    def _generate_recommendations(self, analysis: Dict) -> List[str]:
        """Generate actionable recommendations"""
        recs = []

        if 'demand' in analysis:
            if analysis['demand']['trend'] > 0:
                recs.append("📈 Increase production capacity by 20%")
                recs.append("📦 Negotiate additional supplier contracts")
            else:
                recs.append("📉 Reduce inventory levels gradually")
                recs.append("💰 Consider promotional activities")

        if 'supplier_risk' in analysis:
            risk_level = analysis['supplier_risk']['risk_level']
            if risk_level == 'HIGH':
                recs.append("🚨 Activate contingency suppliers immediately")
                recs.append("📊 Daily monitoring of supplier performance")
            elif risk_level == 'MEDIUM':
                recs.append("⚠️ Increase safety stock by 15%")
                recs.append("📞 Schedule supplier review meeting")

        if 'inventory' in analysis:
            recs.append(f"📦 Set reorder point to {analysis['inventory']['reorder_point']:.0f} units")
            recs.append(f"🎯 Maintain safety stock of {analysis['inventory']['safety_stock']:.0f} units")

        return recs

# Initialize decision support system
decision_support = SupplyChainDecisionSupport(llm_system, ensemble)

# Test scenario
test_scenario = {
    'demand_shock': 500,
    'supplier_issue': {
        'supplier_id': 'S001',
        'reliability': 0.6,
        'delays': 3
    },
    'inventory_constraint': {
        'holding_cost': 2.5,
        'ordering_cost': 100,
        'lead_time': 7
    },
    'market_conditions': 'Volatile due to geopolitical tensions'
}

# Analyze scenario
scenario_analysis = decision_support.analyze_scenario(test_scenario)

print("\n🎯 Scenario Analysis Results:")
print("-" * 50)
for rec in scenario_analysis['recommendations']:
    print(rec)

# ============================================
# CELL 9: Performance Metrics and Evaluation
# ============================================
class PerformanceEvaluator:
    """Comprehensive performance evaluation for supply chain models"""

    def __init__(self):
        self.metrics = {}

    def evaluate_forecasting(self, y_true, y_pred, model_name="model"):
        """Evaluate forecasting performance"""

        metrics = {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
            'r2': 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
        }

        self.metrics[model_name] = metrics
        return metrics

    def evaluate_classification(self, y_true, y_pred, model_name="model"):
        """Evaluate classification performance for risk assessment"""

        from sklearn.metrics import precision_score, recall_score, f1_score

        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='weighted'),
            'recall': recall_score(y_true, y_pred, average='weighted'),
            'f1': f1_score(y_true, y_pred, average='weighted')
        }

        self.metrics[model_name + '_classification'] = metrics
        return metrics

    def create_performance_report(self):
        """Generate comprehensive performance report"""

        report = "=" * 60 + "\n"
        report += "SUPPLY CHAIN AI PERFORMANCE REPORT\n"
        report += "=" * 60 + "\n\n"

        for model_name, metrics in self.metrics.items():
            report += f"Model: {model_name}\n"
            report += "-" * 30 + "\n"
            for metric_name, value in metrics.items():
                report += f"{metric_name.upper()}: {value:.4f}\n"
            report += "\n"

        # Best performing model
        if self.metrics:
            best_model = min(self.metrics.items(),
                           key=lambda x: x[1].get('rmse', float('inf')))
            report += f"🏆 Best Model: {best_model[0]} (RMSE: {best_model[1]['rmse']:.4f})\n"

        return report

# Evaluate all models
evaluator = PerformanceEvaluator()

# Evaluate traditional models
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    evaluator.evaluate_forecasting(y_test, y_pred, name)

# Print performance report
print(evaluator.create_performance_report())

# ============================================
# CELL 10: Save and Export Results
# ============================================
def save_results(models, results, predictions):
    """Save all results and models"""

    # Save model performance
    results_df = pd.DataFrame(results).T
    results_df.to_csv('model_performance.csv')

    # Save predictions
    if predictions is not None:
        pred_df = pd.DataFrame(predictions)
        pred_df.to_csv('predictions.csv')

    # Save model artifacts (for scikit-learn models)
    import joblib
    for name, model in models.items():
        joblib.dump(model, f'{name}_model.pkl')

    print("✅ Results saved successfully!")
    print("Files created:")
    print("  - model_performance.csv")
    print("  - predictions.csv")
    print("  - [model_name]_model.pkl for each model")

    return results_df

# Save results
results_df = save_results(trained_models, evaluation_results, None)

# Display final summary
print("\n" + "=" * 60)
print("SUPPLY CHAIN LLM IMPLEMENTATION COMPLETE!")
print("=" * 60)
print(f"✅ Models trained: {len(trained_models)}")
print(f"✅ Data processed: {len(synthetic_df)} records")
print(f"✅ Features created: {len(feature_cols)}")
print(f"✅ Best RMSE achieved: {results_df['rmse'].min():.4f}")
print("\n📊 Next Steps:")
print("1. Fine-tune LLM on domain-specific supply chain data")
print("2. Implement real-time streaming for production use")
print("3. Add more sophisticated optimization algorithms")
print("4. Integrate with actual ERP/WMS systems")
print("5. Deploy as API service for enterprise use")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.8 MB/s eta 0

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]